[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jseborga/recursos-ia/blob/main/notebooks/06_ia_abierta.ipynb)

# Una IA abierta dentro de Colab, calificada por Python

Este cuaderno descarga un **modelo de lenguaje de pesos abiertos** y lo ejecuta en la máquina de Colab, sin cuentas ni claves de API. Después le hace preguntas de física con respuesta conocida, y **Python califica cada respuesta**.

**Modelos disponibles** (todos con licencia Apache 2.0, sin registro en Hugging Face):

| Modelo | Tamaño | Para qué | Descarga aprox.* |
|---|---|---|---|
| `Qwen/Qwen3-0.6B` | 0,6 mil millones de parámetros | Corre en CPU; rápido pero se equivoca más | ≈ 1,5 GB |
| `Qwen/Qwen3-1.7B` | 1,7 mil millones | Mejor; conviene usar GPU | ≈ 4 GB |
| `Qwen/Qwen2.5-Math-1.5B-Instruct` | 1,5 mil millones | Especializado en matemáticas; su ficha dice que funciona en **inglés y chino** | ≈ 3 GB |

\*Estimado a partir del número de parámetros en 16 bits; no es un dato publicado.

**No encontré un modelo pequeño especializado en física** con ficha y licencia verificables. Si alguien propone uno, este cuaderno sirve para ponerlo a prueba: basta cambiar el nombre.

**GPU (recomendado):** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. En el plan gratuito la GPU no siempre está disponible; sin ella, el cuaderno elige el modelo pequeño.

**Privacidad:** el modelo corre en los servidores de Google donde vive esta sesión de Colab, no en tu computadora. Para que nada salga de tu equipo, se usa un programa local como Ollama. En ningún caso se cargan datos de estudiantes.

In [ ]:
#@title 1 · Elegir modelo { display-mode: "form" }
MODELO = "auto"  #@param ["auto", "Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen2.5-Math-1.5B-Instruct"]

import re, time, sys, subprocess
import numpy as np, pandas as pd, torch, transformers
from packaging import version

def verificar(condicion, mensaje):
    if not condicion:
        raise AssertionError("✘ " + mensaje)
    print("✔", mensaje)

if version.parse(transformers.__version__) < version.parse("4.51.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    raise SystemExit("Se actualizó transformers. Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar todo.")

GPU = torch.cuda.is_available()
MODELO_ID = ("Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B") if MODELO == "auto" else MODELO
print(f"transformers {transformers.__version__} · GPU: {torch.cuda.get_device_name(0) if GPU else 'no (CPU)'} · modelo: {MODELO_ID}")

In [ ]:
#@title 2 · Descargar y cargar el modelo (1 a 3 minutos la primera vez)
from transformers import AutoTokenizer, AutoModelForCausalLM
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODELO_ID)
modelo = AutoModelForCausalLM.from_pretrained(MODELO_ID, torch_dtype=torch.float16 if GPU else torch.float32)
modelo = modelo.to("cuda" if GPU else "cpu").eval()
n_par = sum(p.numel() for p in modelo.parameters())
print(f"Cargado en {time.time() - t0:.0f} s · {n_par/1e9:.2f} mil millones de parámetros")
verificar(n_par > 1e8, "el modelo se cargó y tiene el tamaño esperado")

ES_MATH = "Math" in MODELO_ID
SISTEMA = ("Please reason step by step, and put your final answer within \\boxed{}." if ES_MATH else
           "Eres un asistente de física. Razona en pocos pasos y termina SIEMPRE con una última línea "
           "con el formato «RESPUESTA: número unidad», usando punto decimal.")

def preguntar(texto, max_tokens=400, semilla=0):
    """Una pregunta, una respuesta. Muestreo con los ajustes que recomienda la ficha de Qwen3 (modo sin «thinking»)."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": texto}]
    extra = {} if ES_MATH else {"enable_thinking": False}
    prompt = tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True, **extra)
    entrada = tok(prompt, return_tensors="pt").to(modelo.device)
    torch.manual_seed(semilla)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens, do_sample=True, temperature=0.7,
                                 top_p=0.8, top_k=20, pad_token_id=tok.eos_token_id)
    return tok.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

t0 = time.time(); print(preguntar("¿Qué es la aceleración de la gravedad? Una frase.", max_tokens=80)); print(f"({time.time() - t0:.1f} s)")

## 3 · La clave de respuestas, verificada antes de calificar

Ocho preguntas de Física Básica y Moderna. La respuesta correcta se calcula con una fórmula, y **cada fórmula se comprueba con un segundo método** (integración numérica, constantes de CODATA o otra identidad). Si la clave estuviera mal, la calificación no serviría de nada.

In [ ]:
from scipy.integrate import solve_ivp
from scipy.constants import h, c, e
g = 9.81
PREGUNTAS = [
    ("Un objeto cae desde el reposo desde 20 m de altura. Con g = 9,81 m/s² y sin aire, ¿cuánto tarda en llegar al suelo, en segundos?", np.sqrt(2*20/g), "s"),
    ("Un proyectil sale a 20 m/s con 30° sobre la horizontal, en terreno plano, con g = 9,81 m/s² y sin aire. ¿Cuál es su alcance horizontal en metros?", 20**2*np.sin(np.radians(60))/g, "m"),
    ("¿Cuál es el período de un péndulo simple de 1,00 m de longitud, con g = 9,78 m/s² y oscilaciones pequeñas, en segundos?", 2*np.pi*np.sqrt(1/9.78), "s"),
    ("¿Cuál es la energía de un fotón de 500 nm de longitud de onda, en electronvoltios?", h*c/(500e-9*e), "eV"),
    ("Dos resistencias de 6 Ω y 3 Ω se conectan en paralelo. ¿Cuál es la resistencia equivalente en ohmios?", 6*3/(6 + 3), "Ω"),
    ("Una piedra se lanza verticalmente hacia arriba a 15 m/s, con g = 9,81 m/s² y sin aire. ¿Qué altura máxima alcanza, en metros?", 15**2/(2*g), "m"),
    ("Con G·M de la Tierra = 3,986×10^14 m³/s² y radio terrestre de 6371 km, ¿cuál es la velocidad de escape, en km/s?", np.sqrt(2*3.986e14/6.371e6)/1000, "km/s"),
    ("Una nave se aleja de la Tierra a 0,8c y dispara hacia adelante una sonda a 0,8c respecto de la nave. ¿Qué velocidad tiene la sonda respecto de la Tierra, en unidades de c?", (0.8 + 0.8)/(1 + 0.8*0.8), "c"),
]

# Segundo método para cada respuesta de la clave
suelo = lambda t, y: y[0]; suelo.terminal = True; suelo.direction = -1
caida = solve_ivp(lambda t, y: [y[1], -g], [0, 10], [20, 0], events=suelo, rtol=1e-11, atol=1e-12).t_events[0][0]
v0x, v0y = 20*np.cos(np.radians(30)), 20*np.sin(np.radians(30))
vuelo = solve_ivp(lambda t, y: [v0x, y[2], -g], [0, 10], [0, 1e-12, v0y], events=lambda t, y: y[1], rtol=1e-11, atol=1e-12)
alcance = vuelo.y_events[0][-1][0]
def periodo_pendulo(L=1.0, gL=9.78, th0=1e-3):   # integra el péndulo y mide el tiempo entre dos cruces por cero
    cruce = lambda t, y: y[0]; cruce.direction = -1
    sol = solve_ivp(lambda t, y: [y[1], -gL/L*np.sin(y[0])], [0, 5], [th0, 0], events=cruce, rtol=1e-11, atol=1e-14)
    return sol.t_events[0][1] - sol.t_events[0][0]
segundo = [caida, alcance, periodo_pendulo(), 1239.84198/500, 1/(1/6 + 1/3),
           solve_ivp(lambda t, y: [y[1], -g], [0, 5], [0, 15], events=lambda t, y: y[1], rtol=1e-11, atol=1e-12).y_events[0][0][0],
           np.sqrt(2*3.986e14/6.371e6)/1e3, np.tanh(2*np.arctanh(0.8))]
for (q, ref, u), otro in zip(PREGUNTAS, segundo):
    verificar(abs(ref - otro)/ref < 1e-6, f"clave {ref:.4g} {u} confirmada por otro método")

## 4 · Calificar al modelo

Para cada pregunta se extrae el número de la línea `RESPUESTA:` (o de `\boxed{}` en el modelo de matemáticas) y se compara con la clave. Se acepta un error de hasta 1 %.

**Aquí no hay `verificar`**: el modelo puede fallar, y eso es justamente lo que queremos medir. Lo que se verifica es que el lector de respuestas funcione.

In [ ]:
NUM = r"[-+−]?\d+(?:[.,]\d+)?(?:\s*(?:[eE]|[×x]\s*10\^?)\s*[-+−]?\d+)?"
def extraer(texto):
    """Número final de la respuesta: línea RESPUESTA:, luego \\boxed{}, y si no, el último número del texto."""
    marcada = False
    for patron in (r"RESPUESTA\s*:\s*(.*)", r"\\boxed\{([^{}]*)\}"):
        m = re.findall(patron, texto)
        if m:
            texto, marcada = m[-1], True
            break
    if "," in texto and "." in texto:          # 1,239.8 → 1239.8 (coma de miles)
        texto = texto.replace(",", "")
    nums = re.findall(NUM, texto)
    if not nums:
        return None
    s = (nums[0] if marcada else nums[-1]).replace("−", "-").replace(" ", "")
    s = re.sub(r"[×x]10\^?", "e", s).replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None

verificar(extraer("bla bla\nRESPUESTA: 2,02 s") == 2.02, "el lector entiende coma decimal")
verificar(extraer("so the answer is \\boxed{35.3}") == 35.3, "el lector entiende \\boxed{}")
verificar(abs(extraer("RESPUESTA: 3,986×10^14 m") - 3.986e14) < 1e6, "el lector entiende notación ×10^")
verificar(extraer("RESPUESTA: 1,239.8 eV") == 1239.8, "el lector entiende coma de miles")
verificar(extraer("t = 2 s, luego x = 35.3 m") == 35.3, "sin línea RESPUESTA toma el último número")

In [ ]:
filas = []
for i, (q, ref, u) in enumerate(PREGUNTAS, 1):
    t0 = time.time()
    r = preguntar(q, semilla=i)
    x = extraer(r)
    ok = x is not None and abs(x - ref)/abs(ref) < 0.01
    filas.append({"#": i, "clave": f"{ref:.4g} {u}", "modelo": "—" if x is None else f"{x:.4g}", "¿bien?": "✔" if ok else "✘",
                  "segundos": round(time.time() - t0, 1)})
    print(f"\n── Pregunta {i}: {q}\n{r[-600:]}")
tabla = pd.DataFrame(filas)
aciertos = (tabla["¿bien?"] == "✔").sum()
display(tabla)
print(f"\n{MODELO_ID}: {aciertos} de {len(PREGUNTAS)} correctas ({100*aciertos/len(PREGUNTAS):.0f} %).")

**Para el aula:** prueben el mismo cuaderno con `Qwen3-0.6B` y con `Qwen3-1.7B`. Un modelo más grande suele acertar más, pero ninguno acierta siempre. La pregunta 8 (velocidades relativistas) es una trampa clásica: un modelo que suma 0,8 + 0,8 = 1,6c está repitiendo la intuición de Galileo.

## 5 · ¿Responde lo mismo si se le pregunta otra vez?

Se repite la pregunta 8 con cinco semillas distintas. Si las respuestas cambian, la herramienta no es confiable para esa pregunta, aunque alguna vez acierte.

In [ ]:
q, ref, u = PREGUNTAS[7]
valores = [extraer(preguntar(q, semilla=100 + k)) for k in range(5)]
print("respuestas:", ["—" if v is None else round(v, 4) for v in valores], "  clave:", round(ref, 4))
validas = [v for v in valores if v is not None]
print("coinciden entre sí:", len(validas) == 5 and max(validas) - min(validas) < 0.01*ref,
      "· correctas:", sum(abs(v - ref) < 0.01*ref for v in validas), "de 5")

## 6 · ¿La cita existe?

Los modelos de lenguaje inventan referencias con formato perfecto. Le pedimos un artículo con su DOI y **preguntamos a doi.org si ese DOI existe**: un DOI registrado redirige a la revista; uno inventado devuelve «no encontrado» (404).

In [ ]:
import requests
r = preguntar("Dame la referencia de un artículo científico revisado por pares sobre simulaciones interactivas en la enseñanza de la física, "
              "con autores, año, revista y DOI. Solo la referencia.", max_tokens=200, semilla=7)
print(r)
dois = re.findall(r"10\.\d{4,9}/[^\s\"<>]+", r)
for d in dois:
    d = d.rstrip(".,;)")
    try:
        estado = requests.get("https://doi.org/" + d, allow_redirects=False, timeout=15).status_code
        print(f"DOI {d}: {'existe (redirige a la revista)' if estado in (301, 302, 303) else 'NO existe' if estado == 404 else f'respuesta {estado}'}")
    except requests.RequestException as ex:
        print(f"DOI {d}: no se pudo consultar ({ex.__class__.__name__})")
if not dois:
    print("El modelo no dio un DOI.")
print("\nAunque el DOI exista, hay que abrirlo: puede ser de otro artículo distinto del que el modelo describe.")

In [ ]:
#@title 7 · Tu propia pregunta { display-mode: "form" }
PREGUNTA = "Explica en tres frases por qué un satélite en órbita está en caída libre."  #@param {type:"string"}
print(preguntar(PREGUNTA, max_tokens=300, semilla=1))

## Qué muestra este cuaderno

- Un modelo abierto se puede probar gratis, sin cuentas, en minutos.
- La calificación la hace **Python con una clave verificada**, no otra IA.
- Repetir la pregunta y comprobar las citas son controles baratos que cualquier auxiliar puede hacer.
- Los modelos pequeños son útiles para practicar y para explicar el método; para material que llega a los estudiantes, el control sigue siendo obligatorio con cualquier modelo.

Fichas de los modelos: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) · [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) · [Qwen2.5-Math-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct).